In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("Data/df_negocio.csv")
df.head()


In [ ]:
df["Ratio_Deuda_Ingresos"].describe()

In [ ]:
df["Ratio_Deuda_Ingresos"].value_counts().head(10)

### Amortización alemana

In [ ]:
def amortizacion_alemana(capital, interes_anual, plazo_meses):
    r = interes_anual / 100 / 12
    A = capital / plazo_meses
    
    tabla = []
    saldo = capital
    
    for t in range(1, plazo_meses + 1):
        interes = saldo * r
        cuota = A + interes
        saldo -= A
        
        tabla.append({
            "Mes": t,
            "Amortizacion": round(A, 2),
            "Interes": round(interes, 2),
            "Cuota": round(cuota, 2),
            "Saldo_restante": round(max(saldo, 0), 2)
        })
    
    return pd.DataFrame(tabla)

def reduccion_mensual(capital, interes_anual, plazo_meses):
    r = interes_anual / 100 / 12      
    A = capital / plazo_meses
    return round(r * A, 2)

df_ale = df.dropna(subset=["Monto_Inicial", "Ratio_Interes", "Duracion"]).copy()
df_ale["Duracion"] = df_ale["Duracion"].astype(int)

resultados = {}
resumen = []

for idx, row in df_ale.iterrows():
    cliente = row["ID"]
    capital = row["Monto_Inicial"]       
    interes = row["Ratio_Interes"]
    plazo = int(row["Duracion"])
    
    tabla = amortizacion_alemana(capital, interes, plazo)
    resultados[cliente] = tabla
    
    resumen.append({
        "ID": cliente,
        "Primera_cuota": tabla.loc[0, "Cuota"],
        "Ultima_cuota": tabla.loc[plazo - 1, "Cuota"],
        "Reduccion_mensual": reduccion_mensual(capital, interes, plazo)
    })

resumen_df = pd.DataFrame(resumen)

print("Resumen por cliente:")
print(resumen_df)

primer_cliente = resumen_df.iloc[0]["ID"]
print("\nTabla de amortización para:", primer_cliente)
print(resultados[primer_cliente])


### Amortización Francesa

In [ ]:
def amortizacion_francesa(capital, interes_anual, plazo_meses):
    i = interes_anual / 100 / 12
    cuota = capital * i / (1 - (1 + i) ** (-plazo_meses))

    tabla = []
    saldo = capital

    for t in range(1, plazo_meses + 1):
        interes = saldo * i
        amortizacion = cuota - interes
        saldo -= amortizacion

        tabla.append({
            "Mes": t,
            "Amortizacion": round(amortizacion, 2),
            "Interes": round(interes, 2),
            "Cuota": round(cuota, 2),
            "Saldo_restante": round(max(saldo, 0), 2)
        })

    return pd.DataFrame(tabla)

In [ ]:
df_prest = df.dropna(subset=["Monto_Inicial", "Ratio_Interes", "Duracion"]).copy()
df_prest["Duracion"] = df_prest["Duracion"].astype(int)

i = df_prest["Ratio_Interes"] / 100 / 12
n = df_prest["Duracion"]
capital = df_prest["Monto_Inicial"]

cuota = capital * i / (1 - (1 + i) ** (-n))
primer_interes = capital * i
primer_amortizacion = cuota - primer_interes
ultima_amortizacion = primer_amortizacion * (1 + i) ** (n - 1)   # la amortización crece cada mes
ultimo_interes = cuota - ultima_amortizacion

df_prest["Cuota"] = cuota.round(2)
df_prest["Primer_Interes"] = primer_interes.round(2)
df_prest["Ultimo_Interes"] = ultimo_interes.round(2)
df_prest["Primer_Capital"] = primer_amortizacion.round(2)
df_prest["Ultimo_Capital"] = ultima_amortizacion.round(2)

resumen_df = df_prest[[
    "ID", "Cuota", "Primer_Interes", "Ultimo_Interes", "Primer_Capital", "Ultimo_Capital"
]]

print("Resumen por cliente:")
print(resumen_df)

In [ ]:
cliente = resumen_df.iloc[0]["ID"]
row = df_prest[df_prest["ID"] == cliente].iloc[0]

tabla_cliente = amortizacion_francesa(row["Monto_Inicial"], row["Ratio_Interes"], row["Duracion"])

print("Tabla de amortización para:", cliente)
print(tabla_cliente)

### Modelo de elección de amortización

configura los parámetros del modelo: el umbral para elegir el sistema alemán, las reglas que obligan a elegir el francés, el peso de cada variable y los puntos que recibe cada cliente según sus características.

In [ ]:
UMBRAL_SCORE = 60

KO = {
    "scoring_min": 500,
    "ratio_deuda_max": 0.50,
    "meses_empleo_min": 6,
    "cuota_ingreso_max": 0.50,
    "jornadas_excluidas": ["Desempleado"],
}

PESOS = {
    "Cuota_Ingreso": 18,
    "Scoring_Crediticio": 18,
    "Ratio_Deuda_Ingresos": 14,
    "Meses_Empleo": 8,
    "Tipo_Jornada_Laboral": 8,
    "Ingresos": 8,
    "Duracion": 6,
    "Num_Creditos": 4,
    "Posesion_Hipoteca": 4,
    "Personas_Cargo": 4,
    "Estudios": 4,
    "Fiador": 2,
    "Proposito": 2,
    "Edad": 0,
    "Estado_Civil": 0,
}

assert sum(PESOS.values()) == 100

BANDAS = {
    "Cuota_Ingreso": ([-np.inf, 0.15, 0.25, 0.35, 0.50, np.inf], [1.0, 0.75, 0.40, 0.10, 0.0]),
    "Scoring_Crediticio": ([-np.inf, 500, 600, 700, 800, 900, np.inf], [0.0, 0.15, 0.45, 0.75, 0.90, 1.0]),
    "Ratio_Deuda_Ingresos": ([-np.inf, 0.20, 0.30, 0.40, 0.50, np.inf], [1.0, 0.80, 0.50, 0.20, 0.0]),
    "Meses_Empleo": ([-np.inf, 6, 12, 60, 120, 240, np.inf], [0.0, 0.10, 0.35, 0.60, 0.85, 1.0]),
    "Duracion": ([-np.inf, 12, 24, 36, 48, np.inf], [1.0, 0.80, 0.60, 0.40, 0.20]),
    "Edad": ([-np.inf, 25, 35, 45, 55, 65, np.inf], [0.30, 0.60, 0.85, 1.0, 0.90, 0.70]),
}

PUNTOS_INGRESOS = [0.0, 0.25, 0.50, 0.75, 1.0]
CORTES_INGRESOS = None

CATEGORIAS = {
    "Tipo_Jornada_Laboral": {"Jornada completa": 1.0, "Autónomo": 0.55, "Tiempo parcial": 0.45, "Desempleado": 0.0},
    "Estudios": {"Escolar": 0.40, "Grado Universitario": 0.70, "Máster": 0.90, "Doctorado": 1.0},
    "Estado_Civil": {"Soltero": 0.5, "Casado": 0.5, "Divorciado": 0.5},
    "Posesion_Hipoteca": {0: 0.70, 1: 0.40},
    "Personas_Cargo": {0: 0.70, 1: 0.40},
    "Fiador": {0: 0.40, 1: 0.70},
    "Proposito": {},
    "Num_Creditos": {0: 1.0, 1: 0.90, 2: 0.70, 3: 0.50, 4: 0.30},
}

NEUTRO = 0.5

contiene las funciones que calculan los puntos de cada variable, obtienen la puntuación total de cada cliente sobre 100 y deciden qué sistema de amortización asignarle. Si incumple alguna de las reglas de exclusión, se le asigna el francés, en caso de que no sea asi se decidira mediante la puntuación.

In [ ]:
def ajustar_cortes(df):
    global CORTES_INGRESOS
    q = df["Ingresos"].quantile([0.2, 0.4, 0.6, 0.8]).values
    CORTES_INGRESOS = [-np.inf, *q, np.inf]
    return CORTES_INGRESOS


def _puntos_bandas(serie, bordes, puntos):
    idx = pd.cut(serie, bins=bordes, labels=False)
    pts = np.asarray(puntos, dtype=float)
    return pd.Series(np.where(idx.isna(), NEUTRO, pts[idx.fillna(0).astype(int)]), index=serie.index)


def _puntos_categoria(serie, mapa):
    return serie.map(mapa).fillna(NEUTRO).astype(float)


def calcular_cuota_ingreso(df):
    n = df["Duracion"].where(df["Duracion"] > 0)
    r = df["Ratio_Interes"] / 100 / 12
    cuota1 = df["Monto_Inicial"] / n + df["Monto_Inicial"] * r
    ingreso_mes = (df["Ingresos"] / 12).where(df["Ingresos"] > 0)
    return cuota1 / ingreso_mes


def calcular_score(df):
    if CORTES_INGRESOS is None:
        ajustar_cortes(df)

    d = df.copy()
    d["Cuota_Ingreso"] = calcular_cuota_ingreso(d)
    d["Num_Creditos"] = d["Num_Creditos"].clip(upper=4)

    p = {}
    for col, (bordes, puntos) in BANDAS.items():
        p[col] = _puntos_bandas(d[col], bordes, puntos)
    p["Ingresos"] = _puntos_bandas(d["Ingresos"], CORTES_INGRESOS, PUNTOS_INGRESOS)
    for col, mapa in CATEGORIAS.items():
        p[col] = _puntos_categoria(d[col], mapa)

    puntos = pd.DataFrame(p)[list(PESOS)]
    score = sum(puntos[c] * w for c, w in PESOS.items())
    return score, puntos


def asignar_sistema(df, umbral=UMBRAL_SCORE):
    out = df.copy()
    out["Cuota_Ingreso"] = calcular_cuota_ingreso(out)
    out["Score_Aleman"], _ = calcular_score(out)
    out["Score_Aleman"] = out["Score_Aleman"].round(2)

    condiciones = [
        out["Tipo_Jornada_Laboral"].isin(KO["jornadas_excluidas"]),
        out["Scoring_Crediticio"] < KO["scoring_min"],
        out["Ratio_Deuda_Ingresos"] >= KO["ratio_deuda_max"],
        out["Meses_Empleo"] < KO["meses_empleo_min"],
        out["Cuota_Ingreso"] > KO["cuota_ingreso_max"],
    ]

    motivos = [
        "KO: desempleado",
        "KO: scoring bajo",
        "KO: ratio deuda/ingresos alto",
        "KO: poca antigüedad laboral",
        "KO: cuota alemana inasumible",
    ]

    ko = np.select(condiciones, motivos, default="")
    aleman = (ko == "") & (out["Score_Aleman"] >= umbral)

    out["Sistema"] = np.where(aleman, "aleman", "frances")
    out["Motivo"] = np.where(
        ko != "",
        ko,
        np.where(aleman, "Score >= umbral", "Score < umbral")
    )

    return out

aplica el modelo a los clientes y muestra cuántos reciben cada sistema, los motivos de las decisiones y las puntuaciones obtenidas. Después, compara el porcentaje de impagos de ambos sistemas y prueba distintos umbrales para comprobar cómo cambian los resultados.

In [ ]:
ajustar_cortes(df)
df = asignar_sistema(df)

print(df["Sistema"].value_counts(), "\n")
print(df["Motivo"].value_counts(), "\n")
print(df["Score_Aleman"].describe().round(1))

print("\nImpago por sistema (%):")
print((df.groupby("Sistema")["Impago"].mean() * 100).round(2))

filas = []

for u in range(40, 85, 5):
    t = asignar_sistema(df, umbral=u)
    al = t["Sistema"] == "aleman"

    filas.append({
        "umbral": u,
        "% aleman": round(al.mean() * 100, 1),
        "% impago aleman": round(t.loc[al, "Impago"].mean() * 100, 2) if al.any() else np.nan,
        "% impago frances": round(t.loc[~al, "Impago"].mean() * 100, 2),
    })

pd.DataFrame(filas)

Comprobación de datos faltantes

In [ ]:
print(df[[
    "Ingresos",
    "Monto_Inicial",
    "Duracion",
    "Ratio_Interes",
    "Scoring_Crediticio",
    "Ratio_Deuda_Ingresos",
    "Meses_Empleo",
    "Impago"
]].isnull().sum())

Comprobación de que no hay cuotas no válidas o negativas

In [ ]:
print(df[[
    "Monto_Inicial",
    "Duracion",
    "Ratio_Interes",
    "Ingresos",
    "Cuota_Ingreso"
]].describe().round(2))

print("Cuotas no válidas:", (df["Cuota_Ingreso"].isna()).sum())
print("Cuotas negativas:", (df["Cuota_Ingreso"] < 0).sum())

Comprobación con datos de prueba

In [ ]:
from sklearn.model_selection import train_test_split

entrenamiento, prueba = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print("Clientes para entrenar:", len(entrenamiento))
print("Clientes para probar:", len(prueba))

In [ ]:
ajustar_cortes(entrenamiento)

resultado_prueba = asignar_sistema(prueba)

print("Impago por sistema en los datos de prueba (%):")
print(
    (resultado_prueba.groupby("Sistema")["Impago"].mean() * 100)
    .round(2)
)

print("\nClientes asignados a cada sistema:")
print(resultado_prueba["Sistema"].value_counts())

### Guardado del dataframe

Se recalculan los cortes de ingresos con todos los clientes (la prueba anterior los dejó ajustados solo con el 80 % de entrenamiento), se asigna el sistema a cada cliente y se guarda el resultado para el notebook de valor presente.

In [ ]:
ajustar_cortes(df)
df = asignar_sistema(df)

df[["ID", "Scoring_Crediticio", "Ratio_Deuda_Ingresos", "Cuota_Ingreso", "Score_Aleman", "Sistema", "Motivo"]].head(10)

In [ ]:
df.to_csv("Data/df_negocio_amortizacion.csv", index=False)